# Quantum Canary

Check your qubits before a job, watch them during it, and compare after.

Install: `pip install quantum-canary`  (from a clone of the repository: `pip install -e .`) or un-comment cell below and run

In [2]:
# %pip install -e .

In [3]:
import warnings
warnings.filterwarnings("ignore")

import quantum_canary as canary
from qiskit import QuantumCircuit, transpile

print(canary.banner())


║  ██████╗ ██╗   ██╗ █████╗ ███╗   ██╗████████╗██╗   ██╗███╗   ███╗  ║
║ ██╔═══██╗██║   ██║██╔══██╗████╗  ██║╚══██╔══╝██║   ██║████╗ ████║  ║
║ ██║   ██║██║   ██║███████║██╔██╗ ██║   ██║   ██║   ██║██╔████╔██║  ║
║ ██║▄▄ ██║██║   ██║██╔══██║██║╚██╗██║   ██║   ██║   ██║██║╚██╔╝██║  ║
║ ╚██████╔╝╚██████╔╝██║  ██║██║ ╚████║   ██║   ╚██████╔╝██║ ╚═╝ ██║  ║
║  ╚════▀▀  ╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝  ║
║                                                                    ║
║   ██████╗ █████╗ ███╗   ██╗ █████╗ ██████╗ ██╗   ██╗               ║
║  ██╔════╝██╔══██╗████╗  ██║██╔══██╗██╔══██╗╚██╗ ██╔╝               ║
║  ██║     ███████║██╔██╗ ██║███████║██████╔╝ ╚████╔╝                ║
║  ██║     ██╔══██║██║╚██╗██║██╔══██║██╔══██╗  ╚██╔╝                 ║
║  ╚██████╗██║  ██║██║ ╚████║██║  ██║██║  ██║   ██║                  ║
║   ╚═════╝╚═╝  ╚═╝╚═╝  ╚═══╝╚═╝  ╚═╝╚═╝  ╚═╝   ╚═╝                  ║
║                                                                    ║
║    

## Backend

This uses `fake_fez`, a local copy of IBM's 156-qubit Heron device with its real calibration and noise. It needs no account. Set `USE_IBM = True` to use real hardware instead (your saved IBM Quantum account).

In [4]:
USE_IBM = False

if USE_IBM:
    from qiskit_ibm_runtime import QiskitRuntimeService
    backend = QiskitRuntimeService().backend("ibm_fez")
else:
    from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
    backend = FakeProviderForBackendV2().backend("fake_fez")

print(backend.name, backend.num_qubits, "qubits")

fake_fez 156 qubits


## 1. Before the job

Check the qubits you plan to use, plus a spare. A RED qubit is swapped for the spare. The report also says whether to turn on dynamical decoupling.

In [5]:
before = canary.check(backend, qubits=[13, 14], spares=[12])
before.print()

━━ Pre-job check ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Backend     fake_fez (fake backend, local) · job 2983f774-0bb8-493b-b234-2dc15c84669a
  Priors      live calibration

  Qubit  Role    Health           T1         T2        T2*   Δω/2π kHz       ε_sx  DD
  13     layout  YELLOW     262.5 µs   185.8 µs   231.2 µs     +0.0294   1.49e-04  —
  14     layout  GREEN      225.8 µs   50.71 µs   44.45 µs     +0.0489   1.52e-04  —
  12     spare   GREEN      79.96 µs   75.41 µs   76.09 µs      -0.119   2.80e-04  —

  Use layout  [13, 14]
  Decoupling  not needed on these qubits
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 2. During the job

Your circuits, transpiled onto the checked layout. Here: 20 Bell-state circuits, with a Canary round after every 10.

In [6]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

circuits = [transpile(bell, backend, initial_layout=before.layout) for _ in range(20)]

run = canary.run_monitored(backend, circuits, before, interval=10)
run.print()

━━ Monitored run ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Circuits    20 of yours, a Canary round after every 10 · job 68a62f3b-a5e6-45b6-88db-f5c4350620f6

  Round  Circuits    q13      q14      
  0      0-9         GREEN    GREEN    
  1      10-19       GREEN    GREEN    

  Drift       none since round 0
  Verdict     all rounds GREEN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Your results come back untouched and in order:

In [7]:
print(len(run.results), "results")
print(run.results[0].data.meas.get_counts())

20 results
{'10': 14, '00': 525, '11': 483, '01': 2}


## 3. After the job

Check the same qubits again and compare.

In [8]:
after = canary.check(backend, qubits=before.layout)
canary.compare(before, after).print()

━━ Post-job drift ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Qubit  Before   After         z T1     z T2     z Δω   z ε_sx  Drifted
  13     YELLOW   GREEN        +0.45    +0.09    +0.28    +1.90  —
  14     GREEN    GREEN        -0.71    +0.85    +0.07    +1.41  —

  Verdict     stable: the qubits did not change during the job
              drift = Holm-corrected two-sample z-test at |z| > 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
